In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
import re
import time


In [2]:
session = requests.Session()
standings_urls = {
    'PremierLeague': "https://fbref.com/en/comps/9/Premier-League-Stats",
    'Laliga': "https://fbref.com/en/comps/12/La-Liga-Stats'",
    'Bundesliga': "https://fbref.com/en/comps/20/Bundesliga-Stats",
    'SerieA': "https://fbref.com/en/comps/11/Serie-A-Stats",
    'League1': "https://fbref.com/en/comps/13/ligue-1-Stats",
    # 'BelgianProLeague' : 'https://fbref.com/en/comps/37/Belgian-Pro-League-Stats',
    # 'Eredivisie' : 'https://fbref.com/en/comps/23/Eredivisie-Stats',
    # 'PrimeiraLiga' : 'https://fbref.com/en/comps/32/Primeira-Liga-Stats',
    # 'PremierLeagueWomen' : 'https://fbref.com/en/comps/189/Womens-Super-League-Stats'
}


In [3]:
def get_team_links(b_data, L_soup):
    standings_table = L_soup.select('table.stats_table')[0]
    td_tags = standings_table.find_all('td', {"data-stat": 'team'})

    all_team_names = [a.get_text(strip=True)
                      for td in td_tags for a in td.find_all('a')]

    links = standings_table.find_all('a')
    links = [l.get("href") for l in links]
    links = [l for l in links if '/squads/' in l]
    team_urls = [f"https://fbref.com{l}" for l in links]

    team_ids = [re.search(r'/squads/([^/]+)/', t_url).group(1)
                for t_url in team_urls if re.search(r'/squads/([^/]+)/', t_url)]
    team_name_links = [t_url.split("/")[-1].replace("-Stats", "")
                       for t_url in team_urls]

    return team_urls, all_team_names, team_ids, team_name_links


In [4]:
def get_team_data(data, counter, team_name, fteam, opponent):

    if counter == 0:
        ScoresFixtures = pd.read_html(data.text, match="Scores & Fixtures")[0]
        ScoresFixtures.drop(['Match Report', 'Notes'], axis=1, inplace=True)
        fteam.append(ScoresFixtures)

    elif counter == 1:
        try:
            Shooting = pd.read_html(data.text, match="Shooting", header=1)
            # Shooting[0] = Shooting[0].rename(columns=lambda x: f'{team_name}_' + x)
            Shooting[1] = Shooting[1].rename(columns=lambda x: f'O_' + x)
            fteam.append(Shooting[0].iloc[:-1, 11:-1])
            opponent.append(Shooting[1].iloc[:-1, 11:-1])

        except Exception as e:
            print('The problem is ', e)
    elif counter == 2:
        try:
            Goalkeeping = pd.read_html(
                data.text, match="Goalkeeping", header=1)
            # Goalkeeping[0] = Goalkeeping[0].rename(columns=lambda x: f'{team_name}_' + x)
            Goalkeeping[1] = Goalkeeping[1].rename(columns=lambda x: f'O_' + x)

            Goalkeeping[0].drop('GA.1', axis=1, inplace=True)
            Goalkeeping[1].drop('O_GA.1', axis=1, inplace=True)

            fteam.append(Goalkeeping[0].iloc[:-1, 10:-1])
            opponent.append(Goalkeeping[1].iloc[:-1, 10:-1])

        except Exception as e:
            print('The problem is ', e)
    elif counter == 3:
        try:
            Passing = pd.read_html(data.text, match="Passing", header=1)
            # Passing[0] = Passing[0].rename(columns=lambda x: f'{team_name}_' + x)
            Passing[1] = Passing[1].rename(columns=lambda x: f'O_' + x)

            fteam.append(Passing[0].iloc[:-1, 10:-1])
            opponent.append(Passing[1].iloc[:-1, 10:-1])

        except Exception as e:
            print('The problem is ', e)
    elif counter == 4:
        try:
            PassTypes = pd.read_html(data.text, match="Pass Types", header=1)
            # PassTypes[0] = PassTypes[0].rename(columns=lambda x: f'{team_name}_' + x)
            PassTypes[1] = PassTypes[1].rename(columns=lambda x: f'O_' + x)

            fteam.append(PassTypes[0].iloc[:-1, 10:-1])
            opponent.append(PassTypes[1].iloc[:-1, 10:-1])

        except Exception as e:
            print('The problem is ', e)

    elif counter == 5:
        try:
            GoalShotCreation = pd.read_html(
                data.text, match="Goal and Shot Creation", header=1)
            # GoalShotCreation[0] = GoalShotCreation[0].rename(columns=lambda x: f'{team_name}_' + x)
            GoalShotCreation[1] = GoalShotCreation[1].rename(
                columns=lambda x: f'O_' + x)

            fteam.append(GoalShotCreation[0].iloc[:-1, 10:-1])
            opponent.append(GoalShotCreation[1].iloc[:-1, 10:-1])

        except Exception as e:
            print('The problem is ', e)

    elif counter == 6:
        try:
            DefensiveActions = pd.read_html(
                data.text, match="Defensive Actions", header=1)
            # DefensiveActions[0] = DefensiveActions[0].rename(columns=lambda x: f'{team_name}_' + x)
            DefensiveActions[1] = DefensiveActions[1].rename(
                columns=lambda x: f'O_' + x)

            fteam.append(DefensiveActions[0].iloc[:-1, 10:-1])
            opponent.append(DefensiveActions[1].iloc[:-1, 10:-1])

        except Exception as e:
            print('The problem is ', e)

    elif counter == 7:
        try:
            Possession = pd.read_html(data.text, match="Possession", header=1)
            # Possession[0] = Possession[0].rename(columns=lambda x: f'{team_name}_' + x)
            Possession[1] = Possession[1].rename(columns=lambda x: f'O_' + x)

            fteam.append(Possession[0].iloc[:-1, 10:-1])
            opponent.append(Possession[1].iloc[:-1, 10:-1])

        except Exception as e:
            print('The problem is ', e)

    elif counter == 8:
        try:
            MiscellaneousStats = pd.read_html(
                data.text, match="Miscellaneous Stats", header=1)
            # MiscellaneousStats[0] = MiscellaneousStats[0].rename(columns=lambda x: f'{team_name}_' + x)
            MiscellaneousStats[1] = MiscellaneousStats[1].rename(
                columns=lambda x: f'O_' + x)

            fteam.append(MiscellaneousStats[0].iloc[:-1, 10:-1])
            opponent.append(MiscellaneousStats[1].iloc[:-1, 10:-1])

        except Exception as e:
            print('The problem is ', e)
    return fteam, opponent


In [5]:
def team_stats(team_link, all_team_names, index, team_id, team_name_l):
    print(team_name_l[index])
    data = session.get(team_link)

    soup = BeautifulSoup(data.text, features="lxml")
    mdiv = soup.select('div', {'id': 'content'})[0]
    link = mdiv.find_all('a')
    link = [l.get("href") for l in link]
    link = [f"https://fbref.com" + l for l in link if l != None]

    tables_urls = [
        f'https://fbref.com/en/squads/{team_id[index]}/2023-2024/matchlogs/all_comps/schedule/{
            team_name_l[index]}-Scores-and-Fixtures-All-Competitions',
        f'https://fbref.com/en/squads/{team_id[index]}/2023-2024/matchlogs/all_comps/shooting/{
            team_name_l[index]}-Match-Logs-All-Competitions',
        f'https://fbref.com/en/squads/{team_id[index]}/2023-2024/matchlogs/all_comps/keeper/{
            team_name_l[index]}-Match-Logs-All-Competitions',
        f'https://fbref.com/en/squads/{team_id[index]}/2023-2024/matchlogs/all_comps/passing/{
            team_name_l[index]}-Match-Logs-All-Competitions',
        f'https://fbref.com/en/squads/{team_id[index]}/2023-2024/matchlogs/all_comps/passing_types/{
            team_name_l[index]}-Match-Logs-All-Competitions',
        f'https://fbref.com/en/squads/{team_id[index]}/2023-2024/matchlogs/all_comps/gca/{
            team_name_l[index]}-Match-Logs-All-Competitions',
        f'https://fbref.com/en/squads/{team_id[index]}/2023-2024/matchlogs/all_comps/defense/{
            team_name_l[index]}-Match-Logs-All-Competitions',
        f'https://fbref.com/en/squads/{team_id[index]}/2023-2024/matchlogs/all_comps/possession/{
            team_name_l[index]}-Match-Logs-All-Competitions',
        f'https://fbref.com/en/squads/{team_id[index]}/2023-2024/matchlogs/all_comps/misc/{team_name_l[index]}-Match-Logs-All-Competitions']

    if all(element in link for element in tables_urls[1:]):
        print('True. All links are in this page.')
        TeamStatistics = pd.DataFrame()
        counter = 0

        fteam = []
        opponent = []
        for url in tables_urls:
            res = session.get(url)
            fteam, opponent = get_team_data(
                res, counter, all_team_names[index], fteam, opponent)
            counter += 1
            time.sleep(15)

        fteamdf = pd.concat(fteam, axis=1)
        opponentdf = pd.concat(opponent, axis=1)
        TeamStatistics = pd.concat([fteamdf, opponentdf], axis=1)
        TeamStatistics.insert(9, "Team", all_team_names[index])
        return TeamStatistics
    else:
        return pd.DataFrame()


In [6]:
def get_next_match(df):
    null_counts = df.isnull().sum(axis=1)

    first_row_to_drop = null_counts[null_counts > 295].index[0]
    next_match = df.drop(df.index[first_row_to_drop + 1:])
    return next_match.iloc[-1:, :]


In [8]:
FutureMatche = pd.DataFrame()
NextMatches = []
for l_name, link in standings_urls.items():

    b_data = session.get(link)
    L_soup = BeautifulSoup(b_data.text, features="lxml")
    team_urls, all_team_names, team_id, team_name_l = get_team_links(
        b_data, L_soup)

    for index, team_link in enumerate(team_urls):
        data = session.get(team_link)
        soup = BeautifulSoup(data.text, features="lxml")

        TeamStats = team_stats(team_link, all_team_names,
                               index, team_id, team_name_l)
        nm = get_next_match(TeamStats)
        print(nm.shape)
        NextMatches.append(nm)

        print('*' * 100)
        time.sleep(40)
newdfs = pd.concat(NextMatches, axis=0)
newdfs.to_csv(f"Futures.csv", mode='w', index=False, encoding="utf-8")

# 2:25


Liverpool
True. All links are in this page.
(1, 308)
****************************************************************************************************
Manchester-City
True. All links are in this page.
(1, 308)
****************************************************************************************************
Arsenal
True. All links are in this page.
(1, 308)
****************************************************************************************************
Aston-Villa
True. All links are in this page.
(1, 308)
****************************************************************************************************
Tottenham-Hotspur
True. All links are in this page.
(1, 308)
****************************************************************************************************
Manchester-United
True. All links are in this page.
(1, 308)
****************************************************************************************************
West-Ham-United
True. All links are in this page.
(1, 

In [18]:
data = pd.read_csv('Futures.csv')


In [19]:
data.fillna(0, axis=0, inplace=True)


In [20]:
def preprocess_data(data):
    remove_columns = ['Cmp.1', 'Att.1', 'Cmp%.1', 'Cmp.2', 'Att.2', 'Cmp%.2', 'Cmp.3', 'Att.3', 'Cmp%.3', 'xAG', 'xA', 'CrsPA', 'PrgP',
                      'Live', 'Dead', 'FK', 'TB', 'Sw', 'Crs', 'TI', 'CK', 'In', 'Out', 'Str', 'Off', 'Blocks',
                      'SCA', 'PassDead', 'Def', 'GCA', 'PassDead.1', 'Def.1', 'Def 3rd', 'Mid 3rd', 'Att 3rd', 'Tkl.1', 'Att', 'Tkl%',
                      'Lost', 'Blocks', 'Sh', 'Pass', 'Tkl+Int', 'Err', '2CrdY', 'Int', 'TklW',
                      'Recov', 'Won', 'Lost', 'Won%',
                      'SoTA', 'Save%', 'PKatt', 'PKA', 'PKsv', 'PKm', 'Cmp', 'Att', 'Cmp%',
                      'Att (GK)', 'Thr', 'Launch%', 'AvgLen', 'Att.1', 'Launch%.1', 'AvgLen.1', 'Opp', 'Stp', 'Stp%', '#OPA', 'AvgDist',
                      'O_Cmp.1', 'O_Att.1', 'O_Cmp%.1', 'O_Cmp.2', 'O_Att.2', 'O_Cmp%.2', 'O_Cmp.3', 'O_Att.3', 'O_Cmp%.3', 'O_xAG', 'O_xA', 'O_CrsPA', 'O_PrgP',
                      'Live', 'Dead', 'FK', 'TB', 'Sw', 'Crs', 'TI', 'CK', 'In', 'Out', 'Str', 'Off', 'Blocks',
                      'O_SCA', 'O_PassDead', 'O_Def', 'O_GCA', 'O_PassDead.1', 'O_Def.1', 'O_Def 3rd', 'O_Mid 3rd', 'O_Att 3rd', 'O_Tkl.1', 'O_Att', 'O_Tkl%',
                      'O_Lost', 'O_Blocks', 'O_Sh', 'O_Pass', 'O_Tkl+Int', 'O_Err', 'O_2CrdY', 'O_Int', 'O_TklW',
                      'O_Recov', 'O_Won', 'O_Lost', 'O_Won%',
                      'O_SoTA', 'O_Save%', 'O_PKatt', 'O_PKA', 'O_PKsv', 'O_PKm', 'O_Cmp', 'O_Att', 'O_Cmp%',
                      'O_Att (GK)', 'O_Thr', 'O_Launch%', 'O_AvgLen', 'O_Att.1', 'O_Launch%.1', 'O_AvgLen.1', 'O_Opp',
                      'O_Stp', 'O_Stp%', 'O_#OPA', 'O_AvgDist', 'Time', 'Round', 'G-xG', 'O_G-xG', 'np:G-xG',
                      'O_np:G-xG', 'GCA', 'O_GCA', 'Formation', 'Date', 'Result', 'Captain'
                      ]  # 145 columns
    data.drop(remove_columns, axis=1, inplace=True)
    percentage_columns = [x for x in data.columns if '%' in x]
    data.drop(percentage_columns, axis=1, inplace=True)

    # data['Duplicate'] = data.groupby('Date')['Venue'].transform(
    #     lambda x: x.duplicated(keep='first'))
    # data = data[~data['Duplicate']]
    # data.drop(columns=['Duplicate'], inplace=True)
    return data


In [21]:
def to_matchid(data):
    data['MatchID'] = None

    for i, row in data.iterrows():
        matches = []
        team = row['Team']
        opponent = row['Opponent']
        matches.append(team)
        matches.append(opponent)
        matches.sort()
        data.loc[i, 'MatchID'] = '#' + str(matches[0]) + str(matches[1])
    print(f'The shape of the dataset must be (-1, 169) : {data.shape}')

    return data


In [21]:
data = preprocess_data(data)
# data = to_matchid(data)


In [22]:
unused_cols = ['G/Sh', 'G/SoT', 'Dist', 'PK', 'xG', 'npxG', 'npxG/Sh', 'PSxG', 'PSxG+/-', 'PKatt.1', 'TotDist', 'PrgDist', 'Att.1.1',  'KP', '1/3', 'PPA',  'Att.5', 'FK.1', 'Cmp.5', 'PassLive', 'TO', 'Sh.2', 'Fld', 'PassLive.1', 'TO.1', 'Sh.1', 'Fld.1', 'Tkl', 'Att.6', 'Blocks.1', 'Sh.3', 'Clr', 'Poss.1', 'Touches', 'Def Pen', 'Def 3rd.1', 'Mid 3rd.1', 'Att 3rd.1', 'Att Pen', 'Live.1', 'Att.7', 'Succ', 'Tkld', 'Carries', 'TotDist.1', 'PrgDist.1', 'PrgC', '1/3.1', 'Mis', 'Dis', 'Rec', 'PrgR', 'Fld.2', 'Int.1', 'TklW.1', 'OG', 'Lost.1',
               ]
aunused_cols = []
for co in unused_cols:
    if co == 'Poss.1':
        continue
    conew = 'O_' + co
    aunused_cols.append(conew)
unused_cols.extend(aunused_cols)
data.drop(unused_cols, axis=1, inplace=True)
data.shape


(48, 53)

In [23]:
data.head()


,Comp,Day,Venue,GF,GA,Team,Opponent,xGA,Poss,Attendance,...,O_Off,O_Poss,O_CPA,O_CrdY,O_CrdR,O_Fls,O_Off.1,O_Crs.1,O_PKwon,O_PKcon
0,24,4,0,0.0,0.0,3,258,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,24,0,0,0.0,0.0,5,334,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,24,4,0,0.0,0.0,156,20,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,24,0,0,0.0,0.0,1,1,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,24,4,0,0.0,0.0,12,11,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [24]:
complete_dataset = pd.read_csv('Preprocessed/Preprocessed_data.csv')
# complete_dataset = pd.concat([co_dataset, data], axis=0)
# combined_data.shape

complete_dataset.head()


,Comp,Day,Venue,GF,GA,Team,Opponent,xGA,Poss,Attendance,...,O_In,O_Out,O_Str,O_Off,O_CPA,O_CrdY,O_CrdR,O_Fls,FTR,weight
0,24,0,1,2,0,0,0,0.3,77.0,30415.0,...,2.0,1.0,0.0,6.0,1.0,0.0,0.0,6.0,0.0,0.131153
1,24,1,0,1,1,0,1,0.6,64.0,49108.0,...,0.0,1.0,0.0,3.0,1.0,4.0,1.0,9.0,0.0,0.131579
2,24,0,1,2,1,0,2,0.5,70.0,10419.0,...,2.0,0.0,0.0,0.0,0.0,5.0,0.0,13.0,1.0,0.131817
3,24,0,0,5,0,0,3,0.7,66.0,54172.0,...,3.0,0.0,0.0,2.0,6.0,2.0,1.0,9.0,0.0,0.132486
4,4,2,1,4,0,0,4,0.3,71.0,43500.0,...,2.0,1.0,0.0,3.0,1.0,4.0,0.0,14.0,2.0,0.132679


In [25]:
def ForFutureMatches(merge_data):
    merge_data = pd.get_dummies(merge_data, columns=['Comp', 'Day', 'Venue', 'Team',
                                                     'Opponent', 'Referee'], dtype=int)
    # merge_data['weight'] = 1
    print(f'The shape of the dataset must be (-1, 2169) : {merge_data.shape}')
    return merge_data


In [26]:
dd = ForFutureMatches(complete_dataset)


The shape of the dataset must be (-1, 2169) : (31158, 2169)


In [29]:
pr = np.zeros((48, 2169))
pr = pd.DataFrame(pr, columns=dd.columns)

coln = ['Venue', 'Team', 'Opponent', 'Comp', 'Day']
for co in coln:
    for index, row in data.iterrows():
        comp = row[co]
        competition = f'{co}_' + str(int(comp))
        pr.iloc[index][competition] = 1

pr['weight'] = 1
pr.drop('FTR', axis=1, inplace=True)
pr.head()


,GF,GA,xGA,Poss,Attendance,SoT,xG.1,Saves,CS,CPA,...,Referee_1017,Referee_1021,Referee_1022,Referee_1023,Referee_1025,Referee_1026,Referee_1027,Referee_1029,Referee_1030,Referee_1031
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [30]:
pr.columns.to_list()


['GF',
 'GA',
 'xGA',
 'Poss',
 'Attendance',
 'SoT',
 'xG.1',
 'Saves',
 'CS',
 'CPA',
 'CrdY',
 'CrdR',
 'Fls',
 'Off.1',
 'Crs.1',
 'PKwon',
 'PKcon',
 'O_FK',
 'O_CS',
 'O_Cmp.4',
 'O_Ast',
 'O_Dead',
 'O_TB',
 'O_Sw',
 'O_Crs',
 'O_TI',
 'O_CK',
 'O_In',
 'O_Out',
 'O_Str',
 'O_Off',
 'O_CPA',
 'O_CrdY',
 'O_CrdR',
 'O_Fls',
 'weight',
 'Comp_1',
 'Comp_2',
 'Comp_3',
 'Comp_4',
 'Comp_5',
 'Comp_6',
 'Comp_7',
 'Comp_8',
 'Comp_9',
 'Comp_10',
 'Comp_11',
 'Comp_12',
 'Comp_13',
 'Comp_14',
 'Comp_15',
 'Comp_16',
 'Comp_17',
 'Comp_18',
 'Comp_19',
 'Comp_20',
 'Comp_21',
 'Comp_22',
 'Comp_23',
 'Comp_24',
 'Comp_25',
 'Comp_26',
 'Comp_27',
 'Comp_28',
 'Comp_29',
 'Comp_30',
 'Comp_31',
 'Comp_32',
 'Comp_33',
 'Day_0',
 'Day_1',
 'Day_2',
 'Day_3',
 'Day_4',
 'Day_5',
 'Day_6',
 'Venue_0',
 'Venue_1',
 'Venue_2',
 'Team_0',
 'Team_1',
 'Team_2',
 'Team_3',
 'Team_4',
 'Team_5',
 'Team_6',
 'Team_7',
 'Team_8',
 'Team_9',
 'Team_10',
 'Team_11',
 'Team_12',
 'Team_13',
 'Team

In [129]:
pr.to_csv('matches_for_predict.csv', mode='w', index=False, encoding="utf-8")
